# Lid-driven cavity

Steady incompressible Navier-Stokes at $\mathrm{Re}=100$ on $[0,1]^2$:

$$\mathbf{u}\!\cdot\!\nabla\mathbf{u} + \nabla p - \tfrac{1}{\mathrm{Re}}\nabla^2\mathbf{u}=0,\qquad \nabla\!\cdot\!\mathbf{u}=0.$$

The lid ($y=1$) moves with $u=4x(1-x)$; the other walls are no-slip.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import jax
import jax.numpy as jnp

import pinn
from pinn import operators as op, sampling

%matplotlib inline
jax.config.update("jax_enable_x64", True)   # double precision

In [ ]:
class LidDrivenCavityProblem(pinn.Problem):
    """Steady lid-driven cavity at Re = 100; outputs (u, v, p)."""

    x_min, x_max = 0.0, 1.0
    Re = 100.0
    problem_name = "LidDrivenCavity"
    ref_path = pinn.reference_path("ldc")

    def __init__(self, *, n_pde, n_bc, n_ic=0):
        self.n_pde, self.n_bc, self.n_ic = n_pde, n_bc, n_ic

    def residual_fns(self):
        return {"pde": self.pde_residual, "bc": self.bc_residual}

    def pde_residual(self, model, coords):
        x, y = coords[0], coords[1]
        u_ = lambda c: model(c)[0]
        v_ = lambda c: model(c)[1]
        p_ = lambda c: model(c)[2]
        u, v = u_(coords), v_(coords)
        inv_Re = 1.0 / self.Re
        res_u = (u * op.grad(u_, coords, 0) + v * op.grad(u_, coords, 1)
                 + op.grad(p_, coords, 0)
                 - inv_Re * (op.grad2(u_, coords, 0) + op.grad2(u_, coords, 1)))
        res_v = (u * op.grad(v_, coords, 0) + v * op.grad(v_, coords, 1)
                 + op.grad(p_, coords, 1)
                 - inv_Re * (op.grad2(v_, coords, 0) + op.grad2(v_, coords, 1)))
        res_div = op.grad(u_, coords, 0) + op.grad(v_, coords, 1)
        return jnp.array([res_u, res_v, 2.0 * res_div])

    def bc_residual(self, model, coords):
        x, y = coords[0], coords[1]
        u, v = model(coords)[0], model(coords)[1]
        u_target = jnp.where(jnp.isclose(y, 1.0), 4.0 * x * (1.0 - x), 0.0)
        return jnp.array([u - u_target, v, 0.0])

    def samplers(self):
        return {"pde": self._sample_pde, "bc": self._sample_bc}

    def _sample_pde(self, key, n, *_):
        # 7/8 bulk points, 1/16 near the moving lid, 1/16 in the top corners.
        n_lid, n_corner = n // 16, n // 16
        n_bulk = n - n_lid - n_corner
        k_bulk, k_lx, k_ly, k_l, k_r = jax.random.split(key, 5)
        bulk = jax.random.uniform(k_bulk, (n_bulk, 2), minval=0.0, maxval=1.0)
        lid = jnp.stack([
            jax.random.uniform(k_lx, (n_lid,)),
            jax.random.uniform(k_ly, (n_lid,), minval=0.9, maxval=1.0)], axis=1)
        half = n_corner // 2
        s1 = jax.random.uniform(k_l, (half, 2))
        left = jnp.stack([0.1 * s1[:, 0]**2, 1.0 - 0.1 * s1[:, 1]**2], axis=1)
        s2 = jax.random.uniform(k_r, (half, 2))
        right = jnp.stack([1.0 - 0.1 * s2[:, 0]**2, 1.0 - 0.1 * s2[:, 1]**2], axis=1)
        return jnp.concatenate([bulk, lid, left, right], axis=0)

    def _sample_bc(self, key, n, *_):
        n_corner = n // 16
        n_uniform = n - n_corner
        half = n_corner // 2
        k_pt, k_dim, k_face, k_l, k_r = jax.random.split(key, 5)
        pts = jax.random.uniform(k_pt, (n_uniform, 2), minval=0.0, maxval=1.0)
        face_dim = jax.random.randint(k_dim, (n_uniform,), 0, 2)
        face_val = jax.random.randint(k_face, (n_uniform,), 0, 2).astype(pts.dtype)
        pts = pts.at[jnp.arange(n_uniform), face_dim].set(face_val)
        left = jnp.stack([0.1 * jax.random.uniform(k_l, (half,)),
                          jnp.ones((half,))], axis=1)
        right = jnp.stack([1.0 - 0.1 * jax.random.uniform(k_r, (half,)),
                           jnp.ones((half,))], axis=1)
        return jnp.concatenate([pts, left, right], axis=0)

In [ ]:
cfg = pinn.RunConfig(
    network=lambda key: pinn.SIREN(
        key, LidDrivenCavityProblem, time_dependent=False,
        hidden_dims=(60, 60, 60, 60), periodic_bc=False, n_inputs=2, n_outputs=3,
    ),
    n_pde=2**15, n_bc=2**14,
    residual_sketch=4000, parameter_sketch=4000,
    batch_size=2**12, probe_batch_size=2**8,
    pde_weight=1e-2,
)

In [ ]:
pinn.precompile64(LidDrivenCavityProblem, cfg)

In [ ]:
results = pinn.train64(LidDrivenCavityProblem, cfg)

## Results

The pressure field is defined only up to a constant, so the error panel compares it after removing the mean.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ref = pinn.load_reference("ldc")
x0, x1 = results["plot_x0"], results["plot_x1"]
lx, ly = results["plot_axes"]
extent = [x0[0], x0[-1], x1[0], x1[-1]]

for ch in ref.channels:
    pred = np.array(results["u_pred_plot"][ch])
    exact = np.array(ref.plot_grids[ch])
    err = np.abs(pred - exact)
    rel = np.linalg.norm(pred - exact) / np.linalg.norm(exact)

    fig, axs = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
    for ax, data, title, cmap in zip(
        axs, [pred, exact, err],
        [f"PINN  ${ch}$", f"reference  ${ch}$", f"abs error  (rel $\\ell_2$={rel:.2e})"],
        ["RdBu_r", "RdBu_r", "magma"],
    ):
        im = ax.imshow(data.T, origin="lower", aspect="auto", extent=extent, cmap=cmap)
        ax.set(xlabel=f"${lx}$", ylabel=f"${ly}$", title=title)
        fig.colorbar(im, ax=ax)
    plt.show()